In [0]:
# Source and target tables for the Silver step.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "workspace.sa_market.bronze_rates"
SILVER_TABLE = "workspace.sa_market.silver_rates"

In [0]:
# Converts types, renames columns, drops invalid rates and adds the rand-per-currency rate.
bronze = spark.table(BRONZE_TABLE)

cleaned = (
    bronze.select(
        F.to_date("date").alias("rate_date"),
        F.upper("quote").alias("currency"),
        F.col("rate").cast("double").alias("foreign_per_zar"),
        "source_file",
        "ingested_at",
    )
    .filter(F.col("rate_date").isNotNull() & (F.col("foreign_per_zar") > 0))
    .withColumn("zar_per_foreign", F.round(1 / F.col("foreign_per_zar"), 4))
)

In [0]:
# Keeps only the most recently loaded row for each date and currency.
latest_first = Window.partitionBy("rate_date", "currency").orderBy(F.col("ingested_at").desc())

silver = (
    cleaned.withColumn("row_num", F.row_number().over(latest_first))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
display(silver.orderBy(F.col("rate_date").desc()))

In [0]:
# Writes the cleaned rows to the Silver Delta table.
silver.write.mode("overwrite").saveAsTable(SILVER_TABLE)

In [0]:
%sql
-- Shows the latest rand rate and the year's range per currency.
SELECT currency,
       MAX(rate_date)                 AS latest_day,
       ROUND(MIN(zar_per_foreign), 2) AS lowest,
       ROUND(MAX(zar_per_foreign), 2) AS highest,
       COUNT(*)                       AS row_count
FROM workspace.sa_market.silver_rates
GROUP BY currency
ORDER BY currency